### Notebook to create BLOCK-T394 for LUT in Azimuth with closed loop

Created on: 2025-03-25

Author: Guillem Megias

In [ ]:
from lsst.ts.observing import ObservingBlock, ObservingScript 
from lsst.ts.aos.analysis import build_configuration_schema
import os
import numpy as np

In [ ]:
current_path = os.getcwd()
block_number = 'T394'
program = "BLOCK-T394"
reason = "SITCOM-1490"
note = "LUT_azimuth_closed_loop"
constraints = []

### Define configuration schema

In [ ]:
# Define the configurable properties that we will use in the configuration schema
properties = {
    "filter": {
        "description": "Filter to use.",
        "type": "string",
        "default": "r_57"
    },
    "elevation": {
        "description": "Elevation to use for the aximuthsweep.",
        "type": "number",
        "default": 80
    },
    "exptime": {
        "description": "Exposure time.",
        "type": "number",
        "default": 30.0
    },
    "dofs": {
        "description": "Degrees of freedom to use for the azimuth sweep.",
        "type": "array",
        "items": {
            "type": "integer",
            "minimum": 0,
            "maximum": 49,
        },
        "default": list(range(0, 50))
    },
    "maxiter": {
        "description": "Maximum number of iterations for the closed loop.",
        "type": "integer",
        "default": 1
    }
}

# Build the configuration schema for BLOCK-404
configuration_schema = build_configuration_schema(block_number, properties)
print(configuration_schema)

### Define scripts and block

In [ ]:
closedloop_script = ObservingScript(
    name="maintel/close_loop_lsstcam.py",
    standard=True,
    parameters= dict(
        filter="$filter",
        program="$program",
        reason=reason,
        note=note,
        exposure_time="$exptime",
        max_iter="$maxiter",
        used_dofs="$dofs",
    )
)

stop_tracking_script = ObservingScript(
    name="maintel/stop_tracking.py",
    standard=True,
    parameters = dict()
)

In [ ]:
azimuth_positions = np.arange(-180, 180, 30).tolist()
scripts = []

for azimuth in azimuth_positions:
    track_target_script = ObservingScript(
        name="maintel/track_target.py",
        standard=True,
        parameters = dict(
            track_azel = dict(
                az = azimuth,
                el = '$elevation'
            )
        )
    )

    scripts.append(track_target_script)
    scripts.append(closedloop_script)
scripts.append(stop_tracking_script)

In [ ]:
block = ObservingBlock(
    name = program,
    program = program,
    configuration_schema=configuration_schema,
    scripts = scripts,
)

### Save configurable block

In [ ]:
block.model_dump_json(indent=2)

output_file_path = f'{current_path}/aos/ts_config_ocs/Scheduler/observing_blocks_maintel/AOS/LUTs/{program}.json'

with open(output_file_path, 'w') as file:
    file.write(block.model_dump_json(indent=2))